In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv('titanic.csv')

print(dataset)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

In [ ]:
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsClassifier

features = ['Sex', 'Age', 'Pclass', 'Fare']

# K-Fold 10
kf = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# Menyimpan total error dari semua fold
total_error = 0
total_data = 0

# STEP 2: K-Fold
for fold, (train_index, test_index) in enumerate(kf.split(dataset), 1):

    print(f"\n========== FOLD {fold} ==========")

    # Ambil train dan test
    train_set = dataset.iloc[train_index].copy()
    test_set = dataset.iloc[test_index].copy()

    # STEP 3: Ambil fitur
    train_data = train_set[features].copy()
    test_data = test_set[features].copy()

    # Ubah Sex menjadi angka
    train_data['Sex'] = train_data['Sex'].map({
        'male': 0,
        'female': 1
    })

    test_data['Sex'] = test_data['Sex'].map({
        'male': 0,
        'female': 1
    })

    # Isi missing Age
    # Mean dihitung dari TRAIN
    age_mean = train_data.groupby('Pclass')['Age'].mean()

    train_data['Age'] = train_data['Age'].fillna(train_data.groupby('Pclass')['Age'].transform('mean'))

    # Test menggunakan mean dari TRAIN
    for pclass in age_mean.index:
        test_data.loc[
            (test_data['Pclass'] == pclass) &
            (test_data['Age'].isna()),
            'Age'
        ] = age_mean[pclass]

    # STEP 4: Label
    train_label = train_set['Survived'].copy()
    test_label = test_set['Survived'].copy()

    # STEP 5: Min-Max berdasarkan TRAIN
    train_min = train_data.min()
    train_max = train_data.max()

    train_data = ((train_data - train_min) * (1 - 0) / (train_max - train_min) + 0)

    # STEP 6: Normalisasi TEST
    # Menggunakan min dan max dari TRAIN
    test_data = ((test_data - train_min) * (1 - 0) / (train_max - train_min) + 0)

    # STEP 7: k-NN k=3
    knn = KNeighborsClassifier(n_neighbors=3)

    knn.fit(
        train_data,
        train_label
    )

    prediksi = knn.predict(
        test_data
    )

    # Hitung error pada fold ini
    error = (prediksi != test_label).astype(int)

    jumlah_error = error.sum()
    jumlah_data = len(test_label)

    # Tambahkan ke total
    total_error += jumlah_error
    total_data += jumlah_data

    print("Jumlah error :", jumlah_error)
    print("Jumlah data  :", jumlah_data)
    print("Error Ratio  :", jumlah_error / jumlah_data)



========== FOLD 1 ==========
Jumlah error : 16
Jumlah data  : 90
Error Ratio  : 0.17777777777777778

========== FOLD 2 ==========
Jumlah error : 17
Jumlah data  : 89
Error Ratio  : 0.19101123595505617

========== FOLD 3 ==========
Jumlah error : 16
Jumlah data  : 89
Error Ratio  : 0.1797752808988764

========== FOLD 4 ==========
Jumlah error : 18
Jumlah data  : 89
Error Ratio  : 0.20224719101123595

========== FOLD 5 ==========
Jumlah error : 10
Jumlah data  : 89
Error Ratio  : 0.11235955056179775

========== FOLD 6 ==========
Jumlah error : 10
Jumlah data  : 89
Error Ratio  : 0.11235955056179775

========== FOLD 7 ==========
Jumlah error : 21
Jumlah data  : 89
Error Ratio  : 0.23595505617977527

========== FOLD 8 ==========
Jumlah error : 16
Jumlah data  : 89
Error Ratio  : 0.1797752808988764

========== FOLD 9 ==========
Jumlah error : 23
Jumlah data  : 89
Error Ratio  : 0.25842696629213485

========== FOLD 10 ==========
Jumlah error : 12
Jumlah data  : 89
Error Ratio  : 0.134831460

In [3]:
error_ratio = total_error / total_data

print("\n========== HASIL AKHIR K-FOLD ==========")
print("Total error :", total_error)
print("Total data  :", total_data)
print("Error Ratio :", error_ratio)
print("Error (%)   :", error_ratio * 100, "%")


========== HASIL AKHIR K-FOLD ==========
Total error : 159
Total data  : 891
Error Ratio : 0.17845117845117844
Error (%)   : 17.845117845117844 %
